# 创建SimpleNet对象并设置损失函数、优化器、调度器等训练选项

In [1]:
# 运行准备：按示例代码5.1～5.3组织数据和fe，并使用示例代码5.43定义FeatSet
import os
import csv
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

data_file = "../pybook-data/ch5/imdb_labelled.txt"
df = pd.read_csv(data_file, names=["sentence", "label"], sep="\t", quoting=csv.QUOTE_NONE)
sents = df["sentence"].values
y = df["label"].values

out_dir = "output"
train_file = f"{out_dir}/imdb_labelled_train.csv"
test_file = f"{out_dir}/imdb_labelled_test.csv"
if os.path.exists(train_file) and os.path.exists(test_file):
    print(f'Files exist at "{train_file}" and "{test_file}"')
    df_train, df_test = pd.read_csv(train_file), pd.read_csv(test_file)
    sents_train, sents_test = df_train["sentence"].values, df_test["sentence"].values
    y_train, y_test = df_train["label"].values, df_test["label"].values
else:
    os.makedirs(out_dir, exist_ok=True)
    sents_train, sents_test, y_train, y_test = train_test_split(sents, y, test_size=0.2, random_state=1)
    print(f'训练样本数量：{len(sents_train)}')
    print(f'测试样本数量：{len(sents_test)}')
    df_train = pd.DataFrame({'sentence': sents_train, 'label': y_train})
    df_test = pd.DataFrame({'sentence': sents_test, 'label': y_test})
    df_train.to_csv(train_file, index=False)
    df_test.to_csv(test_file, index=False)

fe = CountVectorizer()
fe.fit(sents_train)
vob = fe.get_feature_names_out()
print(f'词表大小：{len(vob)}')
x_train = fe.transform(sents_train)
x_test = fe.transform(sents_test)
print(f'训练集特征规模：{x_train.shape}')
print(f'测试集特征规模：{x_test.shape}')
import torch
from torch.utils.data import Dataset

class FeatSet(Dataset):
    def __init__(self, feats, labels):
        self.feats = feats
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        dense = self.feats[idx].toarray().squeeze()
        return torch.tensor(dense, dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.long)
# 使用示例代码5.44创建训练集DataLoader
from torch.utils.data import DataLoader
train_ds = FeatSet(fe.transform(sents_train), y_train)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

# 使用示例代码5.45定义SimpleNet
import torch.nn as nn

class SimpleNet(nn.Module):
    def __init__(self, vocab_size, num_class=2):
        super().__init__()
        self.net = nn.Linear(vocab_size, num_class)

    def forward(self, x):
        return self.net(x)

vocab_size = len(vob)

Files exist at "output/imdb_labelled_train.csv" and "output/imdb_labelled_test.csv"
词表大小：2647
训练集特征规模：(800, 2647)
测试集特征规模：(200, 2647)


In [2]:
from transformers import get_linear_schedule_with_warmup

model = SimpleNet(vocab_size)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
model.to(device) #模型 → GPU
criterion = nn.CrossEntropyLoss() #设定损失函数
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=1e-4)
max_epoch = 300
total_steps = len(train_loader) * max_epoch
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
print("可训练参数及参数量:")
print([(name, param.numel()) for name, param in model.named_parameters() if param.requires_grad])

/Users/xinzijie/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


可训练参数及参数量:
[('net.weight', 5294), ('net.bias', 2)]
